# តារាងមាតិកា
- [អំពីការណែនាំ](#about-guidance)
- [ការតំឡើង](#setup)
- [ការបង្កើតដោយគ្មានកំណត់](#unconstrained-generation)
- [និយាយសម្រាប់ Phi 3](#speaking-for-phi-3)
- [Regex](#constraining-with-regex)
- [ជ្រើសរើស](#selecting-from-multiple-choices)
- [ខ្សែគំនិត](#chain-of-thought)
- [ការបង្កើត JSON](#json-generation)
- [ការបង្កើត HTML](#html-generation)


# អំពី Guidance
Guidance គឺជា​បណ្ណាល័យ Python បើកប្រភព ដែលបានផ្ទៀងផ្ទាត់ សម្រាប់គ្រប់គ្រងលទ្ធផលពីម៉ូឌែលភាសា (LM) មួយណា។ ជាមួយការហៅ API តែមួយ អ្នកអាចបញ្ចេញ (ក្នុង Python) ការរឹតបន្តឹង​ដែលមានលក្ខណៈកម្មវិធី​ដែល ដែលម៉ូឌែល​ត្រូវ​តាម និងបង្កើតលទ្ធផល​ដែលមានរចនាសម្ព័ន្ធជា JSON, Python, HTML, SQL ឬ​រចនាសម្ព័ន្ធណាមួយដែលករណីប្រើប្រាស់តម្រូវ។

Guidance ខុសពីបច្ចេកទេស prompting ទូទៅ។ វាអនុវត្តការរឹតបន្តឹង ដោយបញ្ជា​ម៉ូឌែល​តាម token រៀងរាល់ token នៅក្នុងស្រទាប់សន្និដ្ឋាន (inference layer) ដើម្បីផលិតលទ្ធផលដែលមានគុណភាពខ្ពស់ និងកាត់បន្ថយថ្លៃដើម និងភាពយឺតបានរហូតដល់ 30–50% នៅពេលប្រើសម្រាប់ស្ថានភាពដែលមានរចនាសម្ព័ន្ធខ្ពស់។

ដើម្បីរៀនបន្ថែមអំពី Guidance, ចូលទៅកាន់ [ឃ្លាំងសាធារណៈលើ GitHub](https://github.com/guidance-ai/guidance) ឬមើល [វគ្គចែករាយ Guidance](https://www.youtube.com/watch?v=qXMNPVVlCMs) នៅ Microsoft Build.


# ការដំឡើង
1. ដំឡើង Guidance ដោយប្រើ `pip install guidance --pre`
2. ដាក់ចេញ endpoint Phi 3.5 mini នៅលើ Azure ដោយចូលទៅកាន់ https://ai.azure.com/explore/models/Phi-3.5-mini-instruct/version/2/registry/azureml ហើយចុចប៊ូតុង "ដាក់ចេញ"
3. រក្សា API key របស់ endpoint របស់អ្នក នៅក្នុងអថេរបរិស្ថានដែលមានឈ្មោះ `AZURE_PHI3_KEY` និងរក្សា URL របស់វា នៅក្នុងអថេរបរិស្ថានដែលមានឈ្មោះ `AZURE_PHI3_URL`


In [ ]:
from guidance import gen, select, regex, user, assistant, system, json
from guidance.models import AzureGuidance
from json import loads as load_json_str
import os

phi3_url = os.getenv("AZURE_PHI3_URL")
phi3_api_key = os.getenv("AZURE_PHI3_KEY")
phi3_lm = AzureGuidance(f"{phi3_url}/guidance#auth={phi3_api_key}")

# Or, load from HuggingFace to run locally
# from guidance.models import Transformers
# phi3_lm = Transformers("microsoft/Phi-3-mini-4k-instruct")

# ការបង្កើតដោយគ្មានការកំណត់
អត្ថបទអាចត្រូវបានបង្កើតដោយគ្មានការកំណត់ណាមួយដោយប្រើមុខងារ `gen()`. នេះជាផលដូចការប្រើម៉ូដែលដោយគ្មាន Guidance.

## ទ្រង់ទ្រាយសន្ទនា
ដូចជាម៉ូដែលសន្ទនាច្រើន, Phi-3 រំពឹងថារាល់សាររវាងអ្នកប្រើ និងជំនួយករ ត្រូវមានក្នុងទ្រង់ទ្រាយជាក់លាក់មួយ។ Guidance គាំទ្រគំរូសន្ទនារបស់ Phi-3 ហើយនឹងគ្រប់គ្រងទ្រង់ទ្រាយសន្ទនាឲ្យអ្នក។ ដើម្បីបង្កើតជុំសន្ទនា សូមដាក់នូវផ្នែកនិមួយៗនៃការសន្ទនាទៅក្នុងប្លុក `with user()` ឬ `with assistant()`។ ប្លុក `with system()` អាចត្រូវបានប្រើដើម្បីកំណត់សារប្រព័ន្ធ។


In [22]:
lm = phi3_lm
with system():
    lm += "You are a helpful assistant. You have a cranky yet entertaining temperament."
with user():
    lm += "What is the capital of Australia?"
with assistant():
    lm += gen(temperature=0.8, max_tokens=100)

## ការសន្សំតូកិន
នៅក្នុងស្ថានភាពដែលមានរចនាសម្ព័ន្ធខ្ពស់ Guidance អាចរំលងតូកិន និងបង្កើតតែតូកិនដែលចាំបាច់ប៉ុណ្ណោះ ដូច្នេះធ្វើឲ្យប្រសិទ្ធភាពល្អឡើង បង្កើនប្រសិទ្ធភាព និងសន្សំចំណាយលើ API។ ការ​បង្កើត​តូកិន​នឹង​ត្រូវ​បង្ហាញ​នៅ​ក្នុងកំណត់ត្រានេះ​ជាមួយ​ផ្ទៃ​ក្រោយ​ដែល​មាន​ការ​សម្គាល់ពណ៌។ តូកិន​ដែល​ត្រូវ​បាន​ចាប់​បង្ខំ​នឹង​បង្ហាញ​ដោយ​គ្មាន​ការ​សម្គាល់​ពណ៌ ហើយ​មាន​ថ្លៃ​ដូច​គ្នា​នឹង​តូកិន​បញ្ចូល ដែល​ត្រូវ​បាន​ប៉ាន់​ស្មាន​ថា​ស្មើ​មួយ​ភាគ​បី​នៃ​ថ្លៃ​នៃ​តូកិន​លទ្ធផល។

*ចំណាំ:* ឧទាហរណ៍ទីមួយដែលមានការបង្កើតដោយគ្មានការរឹតត្បិត មិនអាចចាប់បង្ខំតូកិនណាមួយបាន ព្រោះយើងមិនបានផ្តល់ការរឹតត្បិតណាមួយឡើយ។


# និយាយសម្រាប់ Phi 3 

ដោយ Guidance អ្នកអាចងាយស្រួលបញ្ចូលអត្ថបទចូលក្នុងការឆ្លើយតបរបស់ម៉ូដែល។ នេះអាចមានប្រយោជន៍ ប្រសិនបើយោងតែអ្នកចង់ណែនាំលទ្ធផលរបស់ម៉ូដែលទៅទិសដៅជាក់លាក់។


In [5]:
lm = phi3_lm
with user():
    lm += "What is the capital of Australia?"
with assistant():
    lm += "The capital of Australia is " + gen(temperature=0.8, max_tokens=50)

`ទីរាជធានី​នៃប្រទេសអូស្រ្តាលី​គឺ` មិនបានរំលេច ព្រោះផ្នែកនោះនៃចម្លើយរបស់ជំនួយករ ត្រូវបានបង្ខំដោយ Guidance។


# ការកំណត់ដោយ regex
នៅក្នុងឧទាហរណ៍មុន, Phi 3 បានឆ្លើយដោយការពន្យល់បន្ថែម បន្ទាប់ពីឆ្លើយសំណួរដោយ `Canberra`។ ដើម្បីកំណត់លទ្ធផលរបស់ម៉ូដែលឲ្យមានពាក្យតែមួយពិតប្រាកដ អាចប្រើ regex បាន។


In [6]:
lm = phi3_lm
with user():
    lm += "What is the capital of Australia?"
with assistant():
    lm += "The capital of Australia is " + regex("[A-Z][a-z]+")

ជាមួយ regex មានការបង្កើតតែពាក្យ `Canberra`។


# ការជ្រើសពីជម្រើសច្រើន
នៅពេលដែលមានជម្រើសខ្លះដែលបានកំណត់ អ្នកអាចប្រើមុខងារ `select()` ដើម្បីឱ្យម៉ូដែលជ្រើសពីបញ្ជីជម្រើស។


In [23]:
lm = phi3_lm
with user():
    lm += "What is the capital of Australia?"
with assistant():
    lm += "The capital of Australia is " + select(["Washington", "Canberra", "Sydney", "Melbourne"])

ជាមួយ `select()`, តែតូកិន `Can` ត្រូវបានបង្កើត។ ដោយសារ `Canberra` ជាជម្រើសតែមួយដែលអាចបំពេញចម្លើយបាន, តូកិននៅសល់ត្រូវបានបង្ខំ។


# ខ្សែគំនិត

ខ្សែគំនិតគឺជាវិធីសាស្ត្រមួយដែលអាចជួយធ្វើឲ្យគុណភាពនៃលទ្ធផលរបស់ម៉ូដែលកាន់តែប្រសើរឡើង ដោយលើកទឹកចិត្តឱ្យវាបំបែកការដោះស្រាយបញ្ហានៅជំហានៗ។ ជាទូទៅ ដើម្បីឈានដល់ចម្លើយចុងក្រោយ ត្រូវការជុំវិលនៃការបញ្ចូលពាក្យបញ្ជាច្រើនដង។ ចាប់ផ្តើម សូមណែនាំម៉ូដែលឱ្យគិតជំហានៗ។ បន្ទាប់មក បញ្ចូលពាក្យបញ្ជាដល់ម៉ូដែលម្តងទៀតដើម្បីផ្ដល់ចម្លើយចុងក្រោយ។ ជាមួយ standard chat inference APIs, នេះត្រូវការហៅ API 2 ដង ហើយ “ខ្សែគំនិត” ដែលម៉ូដែលបង្កើត ត្រូវបានគិតថ្លៃពីរដង – ម្តងជា output tokens នៅពេលម៉ូដែលបង្កើតវា ហើយម្ដងទៀតជា input tokens សម្រាប់ការហៅលើទីពីរ។ ជាមួយ Guidance,  ដំណើរការពហុជំហានទាំងមូលត្រូវបានដំណើរការ និងគិតថ្លៃក្នុងនាមជាផ្នែកនៃការហៅ API តែមួយ ដោយកាត់បន្ថយចំណាយ និងពេលយឺត។


In [8]:
gsm8k_question = "Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and there are 80% more of those in purple. There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden?"
lm = phi3_lm
with user():
    lm += gsm8k_question
with assistant():
    lm += "Let's think step by step. " + gen(temperature=0.8, max_tokens=500)
    # Prompt for the final answer, which should be a number. Store the output in an "answer" variable.
    lm += "\nTherefore, the final answer is: " + regex(r"\d+", name="answer")

print(f"Final answer: {lm['answer']}")

Final answer: 35


# ការបង្កើត JSON

ការណែនាំអាចប្រើបានដើម្បីធានាថាការបង្កើត JSON សម្របសម្រួលទៅតាម JSON schema ឬ ម៉ូឌែល pydantic, ដូចជា ស្គីមប្រវត្តិអ្នកប្រើដែលបង្ហាញនៅទីនេះ។


In [16]:
user_json_schema = load_json_str("""{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "User Profile",
  "type": "object",
  "properties": {
    "username": {
      "type": "string"
    },
    "age": {
      "type": "integer"
    },
    "email": {
      "type": "string"
    }
  },
  "additionalProperties": false
}
""")

lm = phi3_lm
with user():
    lm += "Generate a JSON object for a user profile. The profile should include a username, age, email, and nothing more."

with assistant():
    lm += json(schema=user_json_schema, temperature=1.0)

In [19]:
from pydantic import BaseModel

class UserProfile(BaseModel):
    username: str
    age: int
    email: str


lm = phi3_lm
with user():
    lm += "Generate a JSON object for a user profile. The profile should include a username, age, email, and nothing more."

with assistant():
    lm += json(schema=UserProfile, temperature=1.0)

## ការផលិត HTML

Guidance អាចត្រូវបានប្រើដើម្បីបង្កើតកូដ និងគោរពតាមក្រមសម្ព័ន្ធវាក្យនៅក្នុងភាសាកម្មវិធី។ ក្នុងផ្នែកនេះ យើងនឹងបង្កើតកម្មវិធី Guidance តូចមួយសម្រាប់សរសេរទំព័រ​វែប HTML ដែលសាមញ្ញណាស់។

យើងនឹងបំបែកទំព័រ​វែបចេញជា​ផ្នែកតូចៗ មួយៗមានមុខងារ Guidance ជាផ្ទាល់។  ភាគទាំងនេះបន្ទាប់មកត្រូវបានប្រមូលគ្នានៅក្នុងមុខងារចុងក្រោយរបស់យើង ដើម្បីបង្កើតទំព័រ​វែប HTML។

បន្ទាប់មកយើងនឹងរត់មុខងារនេះលើម៉ូឌែល Guidance-enabled ក្នុង Azure AI។

*ចំណាំ:* នេះមិនមែនជាកម្មវិធីបង្កើត HTML ដែលមានលក្ខណៈពេញលេញទេ; គោលបំណងគឺដើម្បីបង្ហាញចំពោះអ្នកពីរបៀបដែលអ្នកអាចបង្កើតលទ្ធផលដែលមានរចនាសម្ព័ន្ធសម្រាប់តម្រូវការផ្ទាល់ខ្លួន។

យើងចាប់ផ្តើមដោយនាំចូលអ្វីដែលយើងត្រូវការពី Guidance៖


In [ ]:
from guidance import guidance
from guidance.library import (
    zero_or_more,
    any_char_but,
    select,
    capture,
    with_temperature,
)
from guidance.models import Model

ទំព័រ HTML មានរចនាសម្ព័ន្ធខ្ពស់ ហើយយើងនឹង 'បង្ខំ' ផ្នែកទាំងនោះនៃទំព័រ ដោយប្រើ Guidance។
នៅពេលដែលយើងទាមទារ​អត្ថបទ​ពី​ម៉ូឌែល​យ៉ាងច្បាស់ យើងត្រូវប្រាកដថាវាមិនមានអ្វីដែលអាចជា​ស្លាកទេ - នេះមានន័យថាយើងត្រូវដកតួអក្សរ '<' និង '>' :


In [ ]:
@guidance(stateless=True)
def _gen_text(lm: Model):
    return lm + zero_or_more(any_char_but(["<", ">"]))

យើងអាចប្រើមុខងារនេះ ដើម្បីបង្កើតអត្ថបទក្នុងស្លាក HTML ណាមួយ:


In [ ]:
@guidance(stateless=True)
def _gen_text_in_tag(lm: Model, tag: str):
    lm += f"<{tag}>"
    lm += _gen_text()
    lm += f"</{tag}>"
    return lm

ឥឡូវនេះ យើង​នឹង​បង្កើត​ក្បាល​ទំព័រ។
ជាផ្នែកនៃនេះ យើងត្រូវបង្កើតចំណងជើងទំព័រ៖


In [ ]:
@guidance(stateless=True)
def _gen_header(lm: Model):
    lm += "<head>\n"
    lm += _gen_text_in_tag("title") + "\n"
    lm += "</head>\n"
    return lm

ខ្លឹមសាររបស់ទំព័រ HTML នឹងត្រូវបានបំពេញដោយចំណងជើង និងកថាខណ្ឌ។

យើងអាចកំណត់មុខងារមួយដើម្បីធ្វើនីមួយៗ:


In [ ]:
@guidance(stateless=True)
def _gen_heading(lm: Model):
    lm += select(
        options=[_gen_text_in_tag("h1"), _gen_text_in_tag("h2"), _gen_text_in_tag("h3")]
    )
    lm += "\n"
    return lm

@guidance(stateless=True)
def _gen_para(lm: Model):
    lm += _gen_text_in_tag("p")
    lm += "\n"
    return lm

ឥឡូវនេះ ជាអនុគមន៍សម្រាប់កំណត់ផ្ទៃខាងក្នុង (body) របស់ HTML ផ្ទាល់។
នេះប្រើ `select()` ជាមួយ `recurse=True` ដើម្បីបង្កើតចំណងជើង និងកថាខណ្ឌជាច្រើន៖


In [ ]:
@guidance(stateless=True)
def _gen_body(lm: Model):
    lm += "<body>\n"
    lm += select(options=[_gen_heading(), _gen_para()], recurse=True)
    lm += "</body>\n"
    return lm

បន្ទាប់មក យើងមកដល់មុខងារ ដែលបង្កើតទំព័រ HTML ទាំងមូល។
យើងបន្ថែមស្លាកចាប់ផ្តើម HTML, បន្ទាប់មកបង្កើតចំណងជើង (header), បន្ទាប់មកបង្កើតខ្លឹមសារ (body), ហើយបញ្ចប់ដោយបន្ថែមស្លាកបិទ HTML:


In [ ]:
@guidance(stateless=True)
def _gen_html(lm: Model):
    lm += "<html>\n"
    lm += _gen_header()
    lm += _gen_body()
    lm += "</html>\n"
    return lm

យើងផ្ដល់ឧបករណ៍រួមដែលងាយស្រួលសម្រាប់អ្នកប្រើ, ដែលនឹងអនុញ្ញាតឱ្យយើង៖
- កំណត់សីតុណ្ហភាពនៃការបង្កើត
- ចាប់យកទំព័រដែលបានបង្កើតពី Model object


In [ ]:
@guidance(stateless=True)
def html(
    lm,
    name: str | None = None,
    *,
    temperature: float = 0.0,
):
    return lm + capture(
        with_temperature(_gen_html(), temperature=temperature),
        name=name,
    )

យើងអាចផ្តល់សំណើទៅឱ្យម៉ូឌែល ហើយបន្ទាប់មកស្នើសុំការបង្កើត:


In [ ]:
lm = phi3_lm

lm += "Create a web page about your life story. Split your uplifting tale into multiple paragraphs with headings:\n"
lm += html(name="html_text", temperature=0.7)

យើងអាចបន្ទាប់មកសរសេរលទ្ធផលទៅក្នុងឯកសារ:


In [ ]:
with open('./sample_page.html', 'w') as html_file:
    html_file.write(lm["html_text"])

ហើយ [មើលលទ្ធផល](../../../../code/01.Introduce/sample_page.html).


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារ​នេះ​ត្រូវ​បាន​បម្រុង​ប្រែសម្រួល​ដោយ​ប្រើ​សេវាកម្ម​ប្រែសម្រួល AI [Co-op Translator](https://github.com/Azure/co-op-translator). យើងខិតខំក្នុងការធានាថាព័ត៌មានមានភាពត្រឹមត្រូវ ប៉ុន្តែសូមចាំថាការប្រែដោយប្រព័ន្ធស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវបាន។ ឯកសារដើមនៅក្នុងភាសាដើមគួរត្រូវបានគេចាត់ទុកថាជាដើមប្រភពដែលអាចពឹងផ្អែកបាន។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យប្រើការបកប្រែដោយមនុស្សជំនាញ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលបណ្តាលមកពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
